<a href="https://colab.research.google.com/github/samiulislam09/text-to-speech-bengali-models/blob/main/omnivoice_with_our_own_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q "omnivoice[lora]" soundfile librosa pyloudnorm requests
from google.colab import drive, userdata
drive.mount("/content/drive")
import os
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("ok")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.5/168.5 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 110.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fish-speech 2.0.0 requires transformers<=4.57.3, but you have transformers 5.17.0 which is incompatible.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ok


In [ ]:
import sys, json, re, glob, shutil, random, subprocess, zipfile
from pathlib import Path
import torch

assert torch.cuda.is_available(), "No GPU - Runtime > Change runtime type > T4 GPU"
CC = torch.cuda.get_device_capability(0)
BF16_OK = CC[0] >= 8                     # A100/L4 yes, T4 no -> fp16
print(torch.cuda.get_device_name(0), CC, "| bf16:", BF16_OK)
!free -h | head -2

ROOT       = Path("/content/drive/MyDrive/omnivoice-bn-voices")
SRC_OLD    = Path("/content/drive/MyDrive/tts_finetune/source")   # audio saved during the fish-speech attempt
AUDIO_DIR  = ROOT / "source" / "wavs"          # original downloaded wavs
WAV24_DIR  = ROOT / "wav24k"                   # cleaned 24 kHz mono copies
MANIFESTS  = ROOT / "manifests"
TOKEN_DIR  = ROOT / "tokens"
CONFIG_DIR = ROOT / "config"
EXP_DIR    = ROOT / "exp" / "lora"
OUT_DIR    = ROOT / "outputs"
for d in [AUDIO_DIR, WAV24_DIR, MANIFESTS, TOKEN_DIR, CONFIG_DIR, EXP_DIR, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

BASE_MODEL = "k2-fsa/OmniVoice"
TOKENIZER  = "eustlb/higgs-audio-v2-tokenizer"
LANG       = "bn"
VOICES     = ["Kore", "Charon"]      # remove one to use a single voice
OUTPUT_REPO = "sawon19/omnivoice-bn-kore-charon"

# --- training knobs, sized for ~4.5 minutes of audio on a T4 ---
TRAIN_STEPS  = 300       # tiny dataset: each step sees a large slice of it
SAVE_EVERY   = 25        # frequent checkpoints; pick the best by ear
LR           = 5e-5
BATCH_TOKENS = 2048 if not BF16_OK else 4096
MAX_BATCH    = 8
GRAD_ACCUM   = 1

TEST_TEXTS = [
    "আসসালামু আলাইকুম, আজকে আমরা একটা মজার বিষয় নিয়ে কথা বলবো।",
    "আপনার অর্ডারটি কি নিশ্চিত করতে চান?",
    "ঢাকার রাস্তায় আজকে অনেক জ্যাম, তাই একটু দেরি হতে পারে।",
]

Tesla T4 (7, 5) | bf16: False
               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.7Gi       8.9Gi        20Mi       2.4Gi        10Gi


In [ ]:
META = Path("/content/metadata.jsonl")
if META.exists():
    shutil.copy(META, ROOT / "source" / "metadata.jsonl")
rows = [json.loads(l) for l in open(ROOT / "source" / "metadata.jsonl", encoding="utf-8") if l.strip()]
print(len(rows), "rows")

def have(r): p = AUDIO_DIR / Path(r["audio"]).name; return p.exists() and p.stat().st_size > 1000

# a) reuse audio already on Drive from the fish-speech attempt
if (SRC_OLD / "wavs").is_dir():
    for f in (SRC_OLD / "wavs").glob("*.wav"):
        if not (AUDIO_DIR / f.name).exists(): shutil.copy(f, AUDIO_DIR)
# b) unzip wavs.zip if uploaded
if not all(have(r) for r in rows) and Path("/content/wavs.zip").exists():
    !unzip -o -q /content/wavs.zip -d /content/unz -x "__MACOSX/*"
    for f in glob.glob("/content/unz/**/*.wav", recursive=True):
        shutil.copy(f, AUDIO_DIR)
# c) download the rest from their URLs
import requests
for r in rows:
    if not have(r):
        resp = requests.get(r["url"], timeout=60); resp.raise_for_status()
        (AUDIO_DIR / Path(r["audio"]).name).write_bytes(resp.content)

missing = [r["id"] for r in rows if not have(r)]
print("audio files:", len(list(AUDIO_DIR.glob('*.wav'))), "| missing:", missing or "none")

35 rows
audio files: 35 | missing: none


In [ ]:
import numpy as np, librosa, soundfile as sf, pyloudnorm as pyln
from IPython.display import Audio, display

REPLACEMENTS = {
    "২৩শে": "তেইশে", "১৯শে": "উনিশে", "১৭ই": "সতেরোই",
    "২০২৬": "দুই হাজার ছাব্বিশ", "২৪/৭": "টুয়েন্টি ফোর সেভেন",
    "৬,৯৯৯": "ছয় হাজার নয়শো নিরানব্বই", "৯,৯৯৯": "নয় হাজার নয়শো নিরানব্বই",
    "২২": "বাইশ", "ai.selorax.io": "এআই ডট সেলোরাক্স ডট আইও",
}
SKIP_IDS = {"ae5dfff3-7466-4456-bb30-32960e88ee6e-2"}    # 'call ended by user'

import unicodedata
def normalize(t):
    for k in sorted(REPLACEMENTS, key=len, reverse=True):
        t = t.replace(k, REPLACEMENTS[k])
    t = t.replace("—", ",")
    return unicodedata.normalize("NFC", re.sub(r"\s+", " ", t).strip())

SR = 24000
meter = pyln.Meter(SR)
clips = {v: [] for v in VOICES}
for r in rows:
    if r["id"] in SKIP_IDS or r["voice"] not in VOICES:
        continue
    y, _ = librosa.load(AUDIO_DIR / Path(r["audio"]).name, sr=SR, mono=True)
    y, _ = librosa.effects.trim(y, top_db=35)
    y = np.pad(y, int(0.15 * SR))
    y = np.clip(pyln.normalize.loudness(y, meter.integrated_loudness(y), -23.0), -0.99, 0.99)
    out = WAV24_DIR / f"{r['id']}.wav"
    sf.write(out, y, SR, subtype="PCM_16")
    clips[r["voice"]].append({"id": r["id"], "audio_path": str(out),
                              "text": normalize(r["text"]), "duration": len(y) / SR})

for v, c in clips.items():
    print(f"{v}: {len(c)} clips, {sum(x['duration'] for x in c)/60:.1f} min")
left = [x["text"] for c in clips.values() for x in c if re.search(r"[0-9০-৯]", x["text"])]
print("still has digits:", left or "none")

Kore: 24 clips, 3.0 min
Charon: 10 clips, 1.7 min
still has digits: none


In [ ]:
random.seed(0)
train, dev, REFS = [], [], {}
for v, c in clips.items():
    c = sorted(c, key=lambda x: x["id"]); random.shuffle(c)
    dev.append(c[0]); rest = c[1:]
    train += rest
    ref = max([x for x in rest if 5 <= x["duration"] <= 12] or rest, key=lambda x: x["duration"])
    REFS[v] = {"audio": ref["audio_path"], "text": ref["text"]}
    print(f"{v}: train={len(rest)} dev=1 | reference {ref['id']} ({ref['duration']:.1f}s)")
    print("   ", ref["text"]); display(Audio(ref["audio_path"]))

def write_jsonl(items, path):
    with open(path, "w", encoding="utf-8") as f:
        for x in items:
            f.write(json.dumps({"id": x["id"], "audio_path": x["audio_path"],
                                "text": x["text"], "language_id": LANG}, ensure_ascii=False) + "\n")
TRAIN_JSONL, DEV_JSONL = MANIFESTS / "train.jsonl", MANIFESTS / "dev.jsonl"
write_jsonl(train, TRAIN_JSONL); write_jsonl(dev, DEV_JSONL)
json.dump(REFS, open(OUT_DIR / "refs.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print(len(train), "train /", len(dev), "dev")

Kore: train=23 dev=1 | reference 3f307101-d24b-46a7-a21a-3f871c6f24da-1 (10.7s)
    আমাদের Online Private Batch-এর ফি দশ হাজার টাকা। এতে দুই মাসে ত্রিশটার বেশি লাইভ ক্লাস, পাঁচটা ফুল মক টেস্ট আর এক বছরের অ্যাপ অ্যাক্সেস পাবেন। আপনি কি এই ব্যাচেই ভর্তি হতে চাচ্ছেন?


Charon: train=9 dev=1 | reference 7b66e434-f587-47ef-bcb0-1abcf84745f2-0 (7.0s)
    এই event-এ সাধারণত লাঞ্চ আর স্ন্যাকস-এর ব্যবস্থা থাকে, তবে রাজশাহীর menu-টা এখনো final হয়নি। আপনি কি একা আসবেন, নাকি team নিয়ে?


32 train / 2 dev


In [ ]:
GEN = Path("/content/gen.py")
GEN.write_text('''
import json, sys, torch, soundfile as sf
from pathlib import Path
from omnivoice import OmniVoice
cfg = json.load(open(sys.argv[1], encoding="utf-8"))
dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
model = OmniVoice.from_pretrained(cfg["base"], device_map="cuda", dtype=dtype)
if cfg.get("adapter"):
    from omnivoice.utils.lora import load_lora_adapter
    model = load_lora_adapter(model, cfg["adapter"])
sr = model.sampling_rate
out = Path(cfg["out_dir"]); out.mkdir(parents=True, exist_ok=True)
for v, ref in cfg["refs"].items():
    prompt = model.create_voice_clone_prompt(ref_audio=ref["audio"], ref_text=ref["text"])
    if cfg.get("save_prompts"):
        prompt.save(str(Path(cfg["save_prompts"]) / f"{v}_prompt.pt"))
    for i, t in enumerate(cfg["texts"]):
        audio = model.generate(text=t, language=cfg["lang"], voice_clone_prompt=prompt, num_step=32)[0]
        sf.write(str(out / f"{v}_s{i}.wav"), audio, sr)
print("done", cfg["out_dir"])
''', encoding="utf-8")

def generate(out_dir, adapter=None, save_prompts=None):
    cfg = {"base": BASE_MODEL, "adapter": str(adapter) if adapter else None, "refs": REFS,
           "texts": TEST_TEXTS, "lang": LANG, "out_dir": str(out_dir),
           "save_prompts": str(save_prompts) if save_prompts else None}
    cfg_path = Path("/content/gen_cfg.json")
    cfg_path.write_text(json.dumps(cfg, ensure_ascii=False), encoding="utf-8")
    r = subprocess.run([sys.executable, str(GEN), str(cfg_path)], capture_output=True, text=True)
    print(r.stdout[-500:])
    if r.returncode:
        print(r.stderr[-3000:]); raise RuntimeError("generation failed (log above)")

def listen(out_dir):
    for v in VOICES:
        for i, t in enumerate(TEST_TEXTS):
            p = Path(out_dir) / f"{v}_s{i}.wav"
            if p.exists(): print(v, "|", t); display(Audio(str(p)))
print("helper ready")

helper ready


In [ ]:
generate(OUT_DIR / "baseline", save_prompts=OUT_DIR)
listen(OUT_DIR / "baseline")

done /content/drive/MyDrive/omnivoice-bn-voices/outputs/baseline

Kore | আসসালামু আলাইকুম, আজকে আমরা একটা মজার বিষয় নিয়ে কথা বলবো।


Kore | আপনার অর্ডারটি কি নিশ্চিত করতে চান?


Kore | ঢাকার রাস্তায় আজকে অনেক জ্যাম, তাই একটু দেরি হতে পারে।


Charon | আসসালামু আলাইকুম, আজকে আমরা একটা মজার বিষয় নিয়ে কথা বলবো।


Charon | আপনার অর্ডারটি কি নিশ্চিত করতে চান?


Charon | ঢাকার রাস্তায় আজকে অনেক জ্যাম, তাই একটু দেরি হতে পারে।


In [ ]:
train_cfg = {
    "llm_name_or_path": "Qwen/Qwen3-0.6B",
    "audio_vocab_size": 1025, "audio_mask_id": 1024, "num_audio_codebook": 8,
    "audio_codebook_weights": [8, 8, 6, 6, 4, 4, 2, 2],
    "drop_cond_ratio": 0.1, "prompt_ratio_range": [0.0, 0.3], "mask_ratio_range": [0.0, 1.0],
    "language_ratio": 0.8, "use_pinyin_ratio": 0.0, "instruct_ratio": 0.0, "only_instruct_ratio": 0.0,

    "resume_from_checkpoint": None,
    "init_from_checkpoint": BASE_MODEL,

    "use_lora": True, "lora_r": 16, "lora_alpha": 32, "lora_dropout": 0.05, "lora_bias": "none",
    "lora_target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    "lora_modules_to_save": ["audio_embeddings", "audio_heads"],

    "learning_rate": LR, "weight_decay": 0.01, "max_grad_norm": 1.0,
    "steps": TRAIN_STEPS, "seed": 42,
    "warmup_type": "ratio", "warmup_ratio": 0.05, "warmup_steps": 0,

    "attn_implementation": "sdpa",
    "max_sample_tokens": 2000, "min_sample_tokens": 20, "max_batch_size": MAX_BATCH,
    "batch_tokens": BATCH_TOKENS, "gradient_accumulation_steps": GRAD_ACCUM,
    "num_workers": 1,

    "mixed_precision": "bf16" if BF16_OK else "fp16",
    "allow_tf32": BF16_OK,

    "logging_steps": 5, "eval_steps": SAVE_EVERY, "save_steps": SAVE_EVERY,
    "keep_last_n_checkpoints": 20,
}
data_cfg = {
    "train": [{"manifest_path": [str(TOKEN_DIR / "train" / "data.lst")]}],
    "dev":   [{"manifest_path": [str(TOKEN_DIR / "dev" / "data.lst")]}],
}
TRAIN_CFG, DATA_CFG = CONFIG_DIR / "train_lora.json", CONFIG_DIR / "data.json"
TRAIN_CFG.write_text(json.dumps(train_cfg, indent=2)); DATA_CFG.write_text(json.dumps(data_cfg, indent=2))
print(TRAIN_CFG); print(DATA_CFG)

/content/drive/MyDrive/omnivoice-bn-voices/config/train_lora.json
/content/drive/MyDrive/omnivoice-bn-voices/config/data.json


In [ ]:
  !find /content/drive/MyDrive/omnivoice-bn-voices/tokens | head -20

/content/drive/MyDrive/omnivoice-bn-voices/tokens
/content/drive/MyDrive/omnivoice-bn-voices/tokens/train
/content/drive/MyDrive/omnivoice-bn-voices/tokens/train/audios
/content/drive/MyDrive/omnivoice-bn-voices/tokens/train/audios/shard-000000.tar
/content/drive/MyDrive/omnivoice-bn-voices/tokens/train/txts
/content/drive/MyDrive/omnivoice-bn-voices/tokens/train/txts/shard-000000.jsonl
/content/drive/MyDrive/omnivoice-bn-voices/tokens/train/errors.jsonl
/content/drive/MyDrive/omnivoice-bn-voices/tokens/train/data.lst
/content/drive/MyDrive/omnivoice-bn-voices/tokens/dev
/content/drive/MyDrive/omnivoice-bn-voices/tokens/dev/audios
/content/drive/MyDrive/omnivoice-bn-voices/tokens/dev/audios/shard-000000.tar
/content/drive/MyDrive/omnivoice-bn-voices/tokens/dev/txts
/content/drive/MyDrive/omnivoice-bn-voices/tokens/dev/txts/shard-000000.jsonl
/content/drive/MyDrive/omnivoice-bn-voices/tokens/dev/errors.jsonl
/content/drive/MyDrive/omnivoice-bn-voices/tokens/dev/data.lst


In [ ]:
!ls -R /content/drive/MyDrive/omnivoice-bn-voices/tokens | head -20
!wc -l /content/drive/MyDrive/omnivoice-bn-voices/manifests/*.jsonl
!head -c 400 /content/drive/MyDrive/omnivoice-bn-voices/manifests/train.jsonl

/content/drive/MyDrive/omnivoice-bn-voices/tokens:
dev
train

/content/drive/MyDrive/omnivoice-bn-voices/tokens/dev:
audios
data.lst
errors.jsonl
txts

/content/drive/MyDrive/omnivoice-bn-voices/tokens/dev/audios:
shard-000000.tar

/content/drive/MyDrive/omnivoice-bn-voices/tokens/dev/txts:
shard-000000.jsonl

/content/drive/MyDrive/omnivoice-bn-voices/tokens/train:
audios
data.lst
errors.jsonl
    2 /content/drive/MyDrive/omnivoice-bn-voices/manifests/dev.jsonl
   32 /content/drive/MyDrive/omnivoice-bn-voices/manifests/train.jsonl
   34 total
{"id": "00ea59d7-3ed7-4c1d-84c6-dea317e0c218-0", "audio_path": "/content/drive/MyDrive/omnivoice-bn-voices/wav24k/00ea59d7-3ed7-4c1d-84c6-dea317e0c218-0.wav", "text": "দুঃখিত, আমি আপনার কথাটি বুঝতে পারিনি। আপনি কি IELTS কোর্স সম্পর্কে কিছু জানতে চাচ্ছেন?", "language_i

In [ ]:
!python -m omnivoice.scripts.extract_audio_tokens \
    --input_jsonl /content/drive/MyDrive/omnivoice-bn-voices/manifests/train.jsonl \
    --tar_output_pattern "/content/drive/MyDrive/omnivoice-bn-voices/tokens/train/audios/shard-%06d.tar" \
    --jsonl_output_pattern "/content/drive/MyDrive/omnivoice-bn-voices/tokens/train/txts/shard-%06d.jsonl" \
    --tokenizer_path eustlb/higgs-audio-v2-tokenizer \
    --nj_per_gpu 1 --loader_workers 2 --min_num_shards 1 --skip_errors --shuffle True 2>&1 | tail -40
!find /content/drive/MyDrive/omnivoice-bn-voices/tokens | head -20

2026-09-24 07:46:10,458 INFO [extract_audio_tokens.py:341] Input mode: raw JSONL (/content/drive/MyDrive/omnivoice-bn-voices/manifests/train.jsonl)
2026-09-24 07:46:10,483 INFO [extract_audio_tokens.py:427] GPU count: 1, Processes per GPU: 1, Total processes: 1
Extracting Audio Tokens:   0%|          | 0/32 [00:00<?, ?it/s]2026-09-24 07:46:18,832 INFO [extract_audio_tokens.py:548] Submitting tasks... (1 workers)
2026-09-24 07:46:38,916 INFO [_client.py:1025] [Worker 26314] HTTP Request: HEAD https://huggingface.co/eustlb/higgs-audio-v2-tokenizer/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-09-24 07:46:39,012 INFO [_client.py:1025] [Worker 26314] HTTP Request: HEAD https://huggingface.co/eustlb/higgs-audio-v2-tokenizer/resolve/main/preprocessor_config.json "HTTP/1.1 307 Temporary Redirect"
2026-09-24 07:46:39,032 INFO [_client.py:1025] [Worker 26314] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/eustlb/higgs-audio-v2-tokenizer/528e871c2a26c4f0f77

In [ ]:
!python -m omnivoice.scripts.extract_audio_tokens \
    --input_jsonl /content/drive/MyDrive/omnivoice-bn-voices/manifests/dev.jsonl \
    --tar_output_pattern "/content/drive/MyDrive/omnivoice-bn-voices/tokens/dev/audios/shard-%06d.tar" \
    --jsonl_output_pattern "/content/drive/MyDrive/omnivoice-bn-voices/tokens/dev/txts/shard-%06d.jsonl" \
    --tokenizer_path eustlb/higgs-audio-v2-tokenizer \
    --nj_per_gpu 1 --loader_workers 2 --min_num_shards 1 --skip_errors --shuffle True 2>&1 | tail -20
!ls /content/drive/MyDrive/omnivoice-bn-voices/tokens/train /content/drive/MyDrive/omnivoice-bn-voices/tokens/dev

2026-09-24 07:47:27,338 INFO [extract_audio_tokens.py:341] Input mode: raw JSONL (/content/drive/MyDrive/omnivoice-bn-voices/manifests/dev.jsonl)
2026-09-24 07:47:27,367 INFO [extract_audio_tokens.py:427] GPU count: 1, Processes per GPU: 1, Total processes: 1
Extracting Audio Tokens:   0%|          | 0/2 [00:00<?, ?it/s]2026-09-24 07:47:34,101 INFO [extract_audio_tokens.py:548] Submitting tasks... (1 workers)
2026-09-24 07:47:47,192 INFO [extract_audio_tokens.py:575] Processing remaining pending samples...
2026-09-24 07:47:55,521 INFO [_client.py:1025] [Worker 26712] HTTP Request: HEAD https://huggingface.co/eustlb/higgs-audio-v2-tokenizer/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-09-24 07:47:55,614 INFO [_client.py:1025] [Worker 26712] HTTP Request: HEAD https://huggingface.co/eustlb/higgs-audio-v2-tokenizer/resolve/main/preprocessor_config.json "HTTP/1.1 307 Temporary Redirect"
2026-09-24 07:47:55,710 INFO [_client.py:1025] [Worker 26712] HTTP Request: HEAD htt

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
MP = "bf16" if BF16_OK else "fp16"
cmd = ["accelerate", "launch", "--num_processes", "1", "--gpu_ids", "0",
       "--mixed_precision", MP,
       "-m", "omnivoice.cli.train",
       "--train_config", str(TRAIN_CFG),
       "--data_config", str(DATA_CFG),
       "--output_dir", str(EXP_DIR)]
print(" ".join(cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout: print(line, end="")
proc.wait()
print("exit code:", proc.returncode)

accelerate launch --num_processes 1 --gpu_ids 0 --mixed_precision fp16 -m omnivoice.cli.train --train_config /content/drive/MyDrive/omnivoice-bn-voices/config/train_lora.json --data_config /content/drive/MyDrive/omnivoice-bn-voices/config/data.json --output_dir /content/drive/MyDrive/omnivoice-bn-voices/exp/lora
The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.

Fetching 13 files: 100%|██████████| 13/13 [00:00<00:00, 1638.89it/s]

Fetching 13 files: 100%|██████████| 13/13 [00:00<00:00, 1718.38it/s]

Loading weights: 100%|██████████| 313/313 [00:00<00:00, 4265.84it/s]
09/24/2026 07:48:29 - INFO - omnivoice.training.trainer - Loaded Config: TrainingConfig(output_dir='/content/drive/MyDrive/omnivoice-bn-voices/exp/lora', data_config='/content/drive/MyDriv

In [ ]:
src = GEN.read_text(encoding="utf-8")
src = src.replace('''if cfg.get("adapter"):
    from omnivoice.utils.lora import load_lora_adapter
    model = load_lora_adapter(model, cfg["adapter"])''', '''if cfg.get("adapter"):
    loader = None
    for mod in ("omnivoice.utils.lora", "omnivoice.lora", "omnivoice.training.lora", "omnivoice.models.lora"):
        try:
            loader = __import__(mod, fromlist=["load_lora_adapter"]).load_lora_adapter; break
        except (ImportError, AttributeError):
            pass
    if loader:
        model = loader(model, cfg["adapter"])
    else:
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, cfg["adapter"]).merge_and_unload()
    print("adapter loaded via", "omnivoice" if loader else "peft")''')
GEN.write_text(src, encoding="utf-8")
print("gen.py patched")

gen.py patched


In [ ]:
TEST_TEXTS = [
    "আপনার অর্ডারটি আজকে ডেলিভারি হবে, আপনি কি বাসায় থাকবেন?",
    "ধন্যবাদ, আপনার দিনটি শুভ হোক।",
]
generate(OUT_DIR / "custom_ckpt100", adapter=EXP_DIR / "checkpoint-100")
listen(OUT_DIR / "custom_ckpt100")


Loading weights: 100%|██████████| 527/527 [00:04<00:00, 118.47it/s]
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/peft/config.py", line 317, in _get_peft_type
    config_file = hf_hub_download(
        model_id,
        CONFIG_NAME,
        **hf_hub_download_kwargs,
    )
  File "/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py", line 87, in _inner_fn
    validate_repo_id(arg_value)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py", line 135, in validate_repo_id
    raise HFValidationError(
    ...<2 lines>...
    )
huggingface_hub.errors.HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/content/drive/MyDrive/omnivoice-bn-voices/exp/lora/checkpoint-100'. Use `repo_type` argument if needed.

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/content/gen.py",

RuntimeError: generation failed (log above)

In [ ]:
!ls -la {EXP_DIR}
!ls -la {EXP_DIR}/checkpoint-100
!find {EXP_DIR}/checkpoint-100 -maxdepth 2 | head -30
!grep -rn "def .*lora\|save_pretrained\|save_state\|merge_and_unload" /usr/local/lib/python3.13/dist-packages/omnivoice --include=*.py | head -30
!ls /usr/local/lib/python3.13/dist-packages/omnivoice /usr/local/lib/python3.13/dist-packages/omnivoice/utils
!which omnivoice-merge-lora; pip show omnivoice | head -2

total 84
drwx------ 15 root root  4096 Sep 24 08:19 .
drwx------  3 root root  4096 Sep 24 07:27 ..
drwx------  2 root root  4096 Sep 24 07:57 checkpoint-100
drwx------  2 root root  4096 Sep 24 08:00 checkpoint-125
drwx------  2 root root  4096 Sep 24 08:02 checkpoint-150
drwx------  2 root root  4096 Sep 24 08:05 checkpoint-175
drwx------  2 root root  4096 Sep 24 08:07 checkpoint-200
drwx------  2 root root  4096 Sep 24 08:10 checkpoint-225
drwx------  2 root root  4096 Sep 24 07:50 checkpoint-25
drwx------  2 root root  4096 Sep 24 08:12 checkpoint-250
drwx------  2 root root  4096 Sep 24 08:15 checkpoint-275
drwx------  2 root root  4096 Sep 24 08:19 checkpoint-300
drwx------  2 root root  4096 Sep 24 07:52 checkpoint-50
drwx------  2 root root  4096 Sep 24 07:55 checkpoint-75
-rw-------  1 root root  1406 Sep 24 07:48 initial_config.json
drwx------  2 root root  4096 Sep 24 08:15 tensorboard
-rw-------  1 root root 22421 Sep 24 08:19 train.log
total 7190124
drwx------  2 root roo

In [ ]:
from pathlib import Path
import subprocess
import sys
import json
from IPython.display import Audio, display

# Restore gen.py to its original, unpatched state from cell qlcdm5z1UQHx
GEN = Path("/content/gen.py")
GEN.write_text('''
import json, sys, torch, soundfile as sf
from pathlib import Path
from omnivoice import OmniVoice
cfg = json.load(open(sys.argv[1], encoding="utf-8"))
dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
model = OmniVoice.from_pretrained(cfg["base"], device_map="cuda", dtype=dtype)
if cfg.get("adapter"):
    from omnivoice.utils.lora import load_lora_adapter
    model = load_lora_adapter(model, cfg["adapter"])
sr = model.sampling_rate
out = Path(cfg["out_dir"]); out.mkdir(parents=True, exist_ok=True)
for v, ref in cfg["refs"].items():
    prompt = model.create_voice_clone_prompt(ref_audio=ref["audio"], ref_text=ref["text"])
    if cfg.get("save_prompts"):
        prompt.save(str(Path(cfg["save_prompts"]) / f"{v}_prompt.pt"))
    for i, t in enumerate(cfg["texts"]):
        audio = model.generate(text=t, language=cfg["lang"], voice_clone_prompt=prompt, num_step=32)[0]
        sf.write(str(out / f"{v}_s{i}.wav"), audio, sr)
print("done", cfg["out_dir"])
''', encoding="utf-8")
print("gen.py restored to original state.")

# Redefine the generate helper function to correctly load the model from checkpoint
def generate(out_dir, model_source=None, adapter=None, save_prompts=None):
    # If an adapter path is provided and it looks like a full model checkpoint,
    # treat it as the primary model source.
    if isinstance(adapter, Path) and adapter.is_dir() and (adapter / "model.safetensors").exists():
        final_base = str(adapter)
        final_adapter = None # No separate adapter to load, as the checkpoint is the full model
    elif model_source:
        final_base = str(model_source)
        final_adapter = None
    else:
        final_base = BASE_MODEL # Fallback to the original base model
        final_adapter = str(adapter) if adapter else None # Use adapter if explicitly given for non-checkpoint cases

    cfg = {"base": final_base, "adapter": final_adapter, "refs": REFS,
           "texts": TEST_TEXTS, "lang": LANG, "out_dir": str(out_dir),
           "save_prompts": str(save_prompts) if save_prompts else None}

    cfg_path = Path("/content/gen_cfg.json")
    cfg_path.write_text(json.dumps(cfg, ensure_ascii=False), encoding="utf-8")

    print(f"Calling gen.py with base: {cfg['base']}, adapter: {cfg['adapter']}")
    r = subprocess.run([sys.executable, str(GEN), str(cfg_path)], capture_output=True, text=True)
    print(r.stdout[-500:])
    if r.returncode:
        print(r.stderr[-3000:]); raise RuntimeError("generation failed (log above)")

def listen(out_dir):
    import os
    if not Path(out_dir).exists():
        print(f"Output directory {out_dir} does not exist.")
        return

    for v in VOICES:
        for i, t in enumerate(TEST_TEXTS):
            p = Path(out_dir) / f"{v}_s{i}.wav"
            if p.exists(): print(v, "|", t); display(Audio(str(p)))

print("Helper functions 'generate' and 'listen' redefined.")


# Generate audio using the fine-tuned model from checkpoint-100
generate(OUT_DIR / "custom_ckpt100_fixed", adapter=EXP_DIR / "checkpoint-100")
listen(OUT_DIR / "custom_ckpt100_fixed")

gen.py restored to original state.
Helper functions 'generate' and 'listen' redefined.
Calling gen.py with base: /content/drive/MyDrive/omnivoice-bn-voices/exp/lora/checkpoint-100, adapter: None
done /content/drive/MyDrive/omnivoice-bn-voices/outputs/custom_ckpt100_fixed

Kore | আপনার অর্ডারটি আজকে ডেলিভারি হবে, আপনি কি বাসায় থাকবেন?


Kore | ধন্যবাদ, আপনার দিনটি শুভ হোক।


Charon | আপনার অর্ডারটি আজকে ডেলিভারি হবে, আপনি কি বাসায় থাকবেন?


Charon | ধন্যবাদ, আপনার দিনটি শুভ হোক।
